## 6.5 章节实践

通过本章的系统学习，我们掌握了双节点端到端通信、安全配对加密和 MCS 自适应跟踪。现提供以下综合实践练习：

**加密通信 + MCS 自适应联合仿真**，在双节点建链后完成 ECDH 配对和加密传输，同时启用 MCS 自适应跟踪。

要求：

1. 补全 ECDH 配对的发起和消息交换
2. 补全 CRC 反馈传入 LinkQualityTracker

完成后运行 ，观察配对成功率和 MCS 轨迹。

In [ ]:
%%writefile chapter6_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 10.0; n_frames = 40
rng = np.random.default_rng(42)

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

g = SleNode(config=NodeConfig(address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7, enable_encryption=True))
t = SleNode(config=NodeConfig(address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, enable_encryption=True))

# ---- 建链 ----
g.start_advertising(); t.start_scanning()
t.connect(g_addr); g.accept_connection(t_addr, Role.G_NODE)
print(f"Link: G={g.state.name}, T={t.state.name}")

# ==== 1: 补全配对消息交换（2行）====
g_msgs = ______________              # G 发起配对(补全)
for msg in g_msgs:
    t_msgs = ______________          # T 处理配对消息(补全)
for msg in t_msgs:
    g_msgs = g.process_pairing_message(msg)
paired = g.stats["paired"] and t.stats["paired"]
print(f"Pairing: {"OK" if paired else "FAIL"}")

# ---- 加密传输 + MCS 自适应 ----
mcs_hist = []
for i in range(n_frames):
    payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    g.send(payload); tx = g.transmit()
    if tx.iq is None:
        g._qos.arq.on_ack_received(); continue
    rx_iq = _channel_impair(tx.iq, snr_db, "awgn", 6.0,
        0.0, "none", g._tx_config.sps, rng)
    frame = AsyncDataFrame(segment_type=0, data=payload)
    rx = t.receive(rx_iq, len(frame.pack()))
    success = rx.success

    # ==== 2: 补全 MCS 自适应（1行）====
                                                  # 将 CRC 结果传入 LinkQualityTracker(补全)
    suggested = g.recommended_mcs
    if suggested != g.config.mcs_index:
        g.update_mcs(suggested)
        t.update_mcs(suggested)
    mcs_hist.append(g.config.mcs_index)

print(f"MCS range: {min(mcs_hist)} -> {max(mcs_hist)}")
print(f"Final MCS: {mcs_hist[-1]}")
print(f"Adjustments: {sum(1 for i in range(1, len(mcs_hist)) if mcs_hist[i] != mcs_hist[i-1])}")


执行以下命令进行编译并验证结果：


In [ ]:
!python chapter6_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/06.05_answer.txt
